# 템플런 자동 플레이 코드

템플런 플레이 중 주기적으로 플레이 화면을 캡처하여 객체 탐지를 수행하고

객체 탐지 결과에 따라 캐릭터가 수행해야 할 행동을 결정하여

이를 키보드의 입력으로 주어 자동 플레이를 수행

In [2]:
pip install pynput


  Using cached pynput-1.7.7-py2.py3-none-any.whl.metadata (31 kB)
Using cached pynput-1.7.7-py2.py3-none-any.whl (90 kB)


In [1]:
import numpy as np  # NumPy 라이브러리
import cv2  # OpenCV 라이브러리
from PIL import Image  # PIL 라이브러리를 이미지 처리용으로 사용
import win32gui, win32ui, win32con  # Windows GUI 작업을 위한 라이브러리
import os  # OS 라이브러리
from time import sleep  # 시간 지연을 위한 sleep 함수
from pynput.keyboard import Key, Controller  # 키보드 조작을 위한 라이브러리
from ultralytics import YOLOv10  # Ultralytics에서 제공하는 YOLOv10 객체 탐지 모델 사용

In [2]:
# 모델 로딩
model = YOLOv10('v6_nano.pt')  # 학습된 YOLOv10 모델 파일을 로드

# 키보드 컨트롤러 초기화
keyboard = Controller()  # 키보드 조작을 위한 컨트롤러 객체를 생성

# 방향키 매핑
# 0: down, 1: left, 2: right, 3: tilt_left, 4: tilt_right, 5:up
# 3: tilt_left와 4: tilt_right의 경우 키보드 'a', 'd'를 사용하여 캐릭터의 몸을 기울이도록 하는 것이 이상적이지만,
# 오른쪽과 왼쪽이 제대로 검출되지 않는 문제가 가끔 생겨 부득이하게 점프하는 동작으로 바꿈
key_map = {
    0: Key.down,
    1: Key.left,
    2: Key.right,
    3: Key.up,
    4: Key.up,
    5: Key.up
}  # 각 클래스 ID에 따라 키보드 키를 매핑

class WindowCapture:
    def __init__(self, window_name):
        self.hwnd = win32gui.FindWindow(None, window_name)  # 지정된 이름의 윈도우 핸들을 찾기
        if not self.hwnd:
            raise Exception(f"Window not found: {window_name}")  # 윈도우를 찾지 못하면 예외를 발생

        window_rect = win32gui.GetWindowRect(self.hwnd)  # 윈도우의 좌표를 가져오기
        self.w = window_rect[2] - window_rect[0] - 16  # 윈도우 너비 계산
        self.h = window_rect[3] - window_rect[1] - 38  # 윈도우 높이 계산
        self.cropped_x = 8  # 좌측에서 잘라낼 픽셀 수
        self.cropped_y = 30  # 상단에서 잘라낼 픽셀 수

    def get_screenshot(self):
        # 윈도우의 스크린샷을 캡처
        wDC = win32gui.GetWindowDC(self.hwnd)
        dcObj = win32ui.CreateDCFromHandle(wDC)
        cDC = dcObj.CreateCompatibleDC()
        dataBitMap = win32ui.CreateBitmap()
        dataBitMap.CreateCompatibleBitmap(dcObj, self.w, self.h)
        cDC.SelectObject(dataBitMap)
        cDC.BitBlt((0, 0), (self.w, self.h), dcObj, (self.cropped_x, self.cropped_y), win32con.SRCCOPY)

        signedIntsArray = dataBitMap.GetBitmapBits(True)
        img = np.fromstring(signedIntsArray, dtype='uint8')
        img.shape = (self.h, self.w, 4)

        dcObj.DeleteDC()
        cDC.DeleteDC()
        win32gui.ReleaseDC(self.hwnd, wDC)
        win32gui.DeleteObject(dataBitMap.GetHandle())

        img = img[..., :3]  # 알파 채널을 제거하고 RGB 이미지로 변환
        img = np.ascontiguousarray(img)  # 연속된 메모리 블록으로 이미지를 저장
        return img

    def run(self):
        while True:
            img = self.get_screenshot()  # 스크린샷을 캡처
            image = Image.fromarray(img[..., [2, 1, 0]])  # BGR 형식의 이미지를 RGB로 변환
            results = model(source=image, conf=0.5)  # 객체 탐지 모델을 적용
            
            for i, (box, conf, cls) in enumerate(zip(results[0].boxes.xyxy, results[0].boxes.conf, results[0].boxes.cls)):
                ymin, ymax = box[1], box[3]
                if 450 <= ymax <= 650:  # 객체의 y좌표가 지정된 범위 내에 있는지 확인
                    print(ymax)
                    key = key_map[int(cls)]  # 매핑된 키를 가져오기
                    if isinstance(key, Key):
                        # 특수키 조작 (up, down, right, left)
                        keyboard.press(key)
                        keyboard.release(key)
                    else:
                        # 일반 문자키 조작 ('a', 'd')
                        keyboard.press(key)
                        keyboard.release(key)
            
            sleep(0.01)  # 0.01초 간격으로 루프를 반복

# 메인 실행
if __name__ == "__main__":
    wc = WindowCapture("BlueStacks App Player")  # BlueStacks 앱 플레이어의 윈도우를 캡처하기 위한 인스턴스를 생성
    wc.run()  # 캡처와 객체 탐지를 반복 실행


C:\Users\user\anaconda3\envs\dl\Lib\site-packages\ultralytics\nn\tasks.py:733: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")
C:\U


0: 640x384 (no detections), 223.0ms
Speed: 14.1ms preprocess, 223.0ms inference, 22.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 157.8ms
Speed: 4.9ms preprocess, 157.8ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 144.5ms
Speed: 2.5ms preprocess, 144.5ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 133.7ms
Speed: 6.0ms preprocess, 133.7ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 143.3ms
Speed: 3.5ms preprocess, 143.3ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 229.9ms
Speed: 5.9ms preprocess, 229.9ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 175.7ms
Speed: 5.8ms preprocess, 175.7ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 155.0ms
Speed: 4.9ms pre

KeyboardInterrupt: 